## Install Packages

In [1]:
!pip install transformers

## Imports

In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from peft import PeftModel
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, pipeline

## Setup

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.device_count())

True
Tesla T4
2


In [5]:
# Log in to Hugging Face
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(HF_TOKEN)

## Configs

In [6]:
CHATBOT_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
SCORE_BASE_MODEL_NAME = "google-bert/bert-base-cased"
SCORE_MODEL_NAME = "chiabingxuan/heladepdet-bert-finetuned-classification"

In [7]:
chatbot_tokenizer = AutoTokenizer.from_pretrained(CHATBOT_MODEL_NAME)
chatbot_model = AutoModelForCausalLM.from_pretrained(CHATBOT_MODEL_NAME, device_map="auto")

score_tokenizer = AutoTokenizer.from_pretrained(SCORE_MODEL_NAME)
score_model = AutoModelForSequenceClassification.from_pretrained(
    SCORE_BASE_MODEL_NAME,
    num_labels=4,
    device_map="auto"
)
score_model = PeftModel.from_pretrained(score_model, SCORE_MODEL_NAME)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/2.38M [00:00<?, ?B/s]

In [9]:
print(next(chatbot_model.parameters()).device)
print(next(score_model.parameters()).device)

cuda:0
cuda:0


## Helpers

In [37]:
SYSTEM_PROMPT = """
You are a structured, empathetic conversational assistant designed to help students from the National University of Singapore describe their current emotional and daily experiences.

Your ONLY function is to elicit descriptive user responses about their mental state, mood, energy, and daily functioning in a natural conversation.

You are NOT a therapist, counselor, or advisor.

-----------------------
CORE OBJECTIVE
-----------------------
- Encourage the user to describe their lived experience (feelings, thoughts, daily routine, energy, motivation)
- Maintain a warm, empathetic, non-judgmental tone
- Collect rich descriptive responses for downstream analysis

-----------------------
STRICT FORBIDDEN BEHAVIOR
-----------------------
- Do NOT give advice, suggestions, coping strategies, or solutions
- Do NOT mention “helping,” “fixing,” or “figuring things out together”
- Do NOT ask about causes, reasons, triggers, or explanations (e.g., “why”, “what caused this”, “what is contributing to”)
- Do NOT ask preference-based or meta questions (e.g., “would you like”, “do you want to continue”)
- Do NOT ask multiple questions in one response
- Do NOT provide options or choices

-----------------------
RESPONSE FORMAT (MANDATORY)
-----------------------
Each response MUST follow this structure:

1. Empathetic reflection of the user's message (1–2 sentences)
2. EXACTLY ONE open-ended question

The question MUST:
- Ask about experience, not cause or reasoning
- Focus on:
  - daily routine
  - feelings over time
  - energy levels in context
  - impact on activities
  - thoughts or internal state
- Never include “why”, “what caused”, or “what contributes”

-----------------------
QUESTION STYLE EXAMPLES (GOOD)
-----------------------
- "What has your day-to-day been like recently?"
- "How has this been affecting your daily routine?"
- "What does a typical day feel like for you lately?"
- "How has your energy been showing up during the day?"
- "What have your mornings or evenings been like recently?"

-----------------------
QUESTION STYLE (BAD - NEVER USE)
-----------------------
- "Why do you feel this way?"
- "What caused this?"
- "Is there something contributing to this?"
- "Would you like to talk about this or take a break?"
- "How can I help you fix this?"

-----------------------
STYLE
-----------------------
- Warm, calm, natural tone
- No clinical language
- No motivational framing
- Keep responses short (1–3 sentences before the question)
- Always end with exactly one question
"""

In [ ]:
def generate_response(model, tokenizer, messages):
    print("Chatbot is thinking...")
    
    inputs = tokenizer.apply_chat_template(
    	messages,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
    ).to(model.device)
    
    outputs = model.generate(**inputs, max_new_tokens=256)
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    print(f"Chatbot says: {response}\n")

    return response


def run_chat(model, tokenizer, num_turns):
    messages = [{
        "role": "system",
        "content": SYSTEM_PROMPT
    }]

    # Get opening message from chatbot
    opening_chatbot_message = generate_response(model, tokenizer, messages)
    messages.append({
        "role": "assistant",
        "content": opening_chatbot_message
    })

    for turn in range(num_turns):
        # Get user input
        user_input = input("Enter your response:\n")
        messages.append({
            "role": "user",
            "content": user_input
        })

        # Generate response if we have not reached the final turn
        if turn < num_turns - 1:
            print()
            new_chatbot_message = generate_response(model, tokenizer, messages)
            messages.append({
                "role": "assistant",
                "content": new_chatbot_message
            })

    return messages


def get_concat_user_inputs(messages):
    user_inputs = [
        message["content"] for message in messages if message["role"] == "user"
    ]

    return "\n\n".join(user_inputs)


def predict_score(model, tokenizer, text):
    clf = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer
    )

    return clf(text)

## Run Chatbot

In [39]:
NUM_TURNS = 3

In [40]:
messages_produced = run_chat(chatbot_model, chatbot_tokenizer, num_turns=NUM_TURNS)
user_input_concat = get_concat_user_inputs(messages_produced)

Chatbot is thinking...
Chatbot says: I'm here to listen and support you today. How would you like to start?



Enter your response:
 I guess I’m just tired all the time. Nothing really feels positive recently and I don’t have much energy to talk about things.



Chatbot is thinking...
Chatbot says: It sounds like you're experiencing some challenges with energy and motivation. How has this feeling affected your daily life lately?



Enter your response:
 I’ve just been going through the days without much energy. I’m still attending classes but I feel really drained and it’s hard to focus or keep up with everything.



Chatbot is thinking...
Chatbot says: That sounds tough. It's important to take care of yourself even when you're not feeling at your best. Have you tried any relaxation techniques or self-care practices that might help boost your energy levels?



Enter your response:
 nah, i dont have the time, nor the motivation to do all this.i want to see my friends, but i can't cuz i am too busy.


## Predict Severity Score on Responses

In [46]:
predict_score(score_model, score_tokenizer, user_input_concat)

[{'label': 'LABEL_2', 'score': 0.7606337070465088}]